# Emotion-Aware Music Recommendation from Text
### A follow-up study extending my final-year project, *Harmony Hub*

**Background.** My final-year project, Harmony Hub, designed a wellbeing app that recommends music, films and other content based on how the user feels. The design detected emotion from a short piece of text the user types, then recommended matching content. That report described the method but did not compare models or measure results.

**This notebook does that missing evaluation.** It asks three questions:

1. **How accurately can a model detect emotion from short, informal text?** I compare a lexicon tool (VADER), three classic machine-learning models (Naive Bayes, Linear SVM, Logistic Regression) and a neural network (BiLSTM). There is also an optional transformer model (DistilBERT).
2. **Which emotions are hardest to tell apart, and why?** Using a confusion matrix and the examples the model gets wrong.
3. **Can the detected emotion drive song recommendations?** Using the arousal–valence model of emotion to map each emotion to Spotify audio features (valence and energy).

**Data**
- *Emotion* dataset (Saravia et al., 2018, EMNLP): about 20,000 English sentences labelled with six emotions (sadness, joy, love, anger, fear, surprise), already split into train, validation and test.
- *Spotify Tracks* dataset: about 114,000 tracks with audio features and genre.

**How to run:** in Google Colab, click **Runtime → Run all**. For the optional DistilBERT section, first set **Runtime → Change runtime type → T4 GPU**.

## 1. Setup
Install and import the libraries. Colab already has most of them.

In [ ]:
!pip install -q datasets vaderSentiment

In [ ]:
import re, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

# Fix random seeds so results are repeatable
SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 2. Load the emotion dataset
The dataset comes already split into **train** (for learning), **validation** (for choosing settings) and **test** (only used once, at the end, for the final score). Keeping the test set untouched until the end is what makes the final numbers fair.

In [ ]:
LABELS = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

def load_emotion_data():
    try:
        from datasets import load_dataset
        ds = load_dataset('dair-ai/emotion')
        splits = {name: ds[name].to_pandas() for name in ['train', 'validation', 'test']}
    except Exception as e:
        print('datasets library failed, trying direct download:', e)
        base = 'https://huggingface.co/datasets/dair-ai/emotion/resolve/main/split/'
        files = {'train': 'train-00000-of-00001.parquet',
                 'validation': 'validation-00000-of-00001.parquet',
                 'test': 'test-00000-of-00001.parquet'}
        splits = {name: pd.read_parquet(base + f) for name, f in files.items()}
    for name, df in splits.items():
        df['label'] = df['label'].map(lambda i: LABELS[i])
    return splits['train'], splits['validation'], splits['test']

train_df, val_df, test_df = load_emotion_data()
print(f'Train: {len(train_df)}  Validation: {len(val_df)}  Test: {len(test_df)}')
train_df.head()

### 2.1 How balanced are the emotions?

In [ ]:
counts = train_df['label'].value_counts().reindex(LABELS)
ax = counts.plot(kind='bar', color='steelblue', figsize=(7, 3.5))
ax.set_title('Training examples per emotion'); ax.set_ylabel('Count')
for i, v in enumerate(counts): ax.text(i, v, str(v), ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

print((counts / counts.sum() * 100).round(1).astype(str) + ' %')
print('Average sentence length (words):', round(train_df['text'].str.split().str.len().mean(), 1))

**What this shows:** the classes are **imbalanced**. *Joy* and *sadness* make up most of the data, while *surprise* and *love* are rare. A model could score well on plain accuracy just by being good at the common emotions. So I also report **macro F1**, which gives every emotion equal weight. Macro F1 is my main metric.

## 3. Text preprocessing
My dissertation planned the standard pipeline: **lowercasing, tokenisation, stopword removal and lemmatisation**. I test whether stopword removal actually helps here. Standard stopword lists delete words like *not*, *no* and *very*, which can flip or strengthen an emotion ("I am **not** happy").

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP = set(stopwords.words('english'))
NEGATIONS = {'not', 'no', 'nor', 'never', "don't", "didn't", "isn't", "wasn't", "can't", "couldn't", "won't"}

def clean(text, remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"[^a-z' ]", ' ', text)          # keep letters and apostrophes only
    tokens = text.split()                            # tokenisation
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP]
    tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]  # lemmatisation: 'feels' -> 'feel', 'was' -> 'be'
    return ' '.join(tokens)

example = "i didn't feel happy at all, i was feeling so lonely"
print('Original:            ', example)
print('Keep stopwords:      ', clean(example, remove_stopwords=False))
print('Remove stopwords:    ', clean(example, remove_stopwords=True))
print('Negation words in standard stopword list:', sorted(NEGATIONS & STOP))

In [ ]:
for df in (train_df, val_df, test_df):
    df['clean_keep'] = df['text'].apply(lambda t: clean(t, remove_stopwords=False))
    df['clean_nostop'] = df['text'].apply(lambda t: clean(t, remove_stopwords=True))
print('Done.')

## 4. Baseline: VADER (lexicon tool from my dissertation)
VADER scores how **positive or negative** a sentence is, using a fixed dictionary of words. It cannot tell the six emotions apart. So to compare it fairly, I group the emotions into **positive** (joy, love, surprise) and **negative** (sadness, anger, fear) and check how often VADER gets the group right.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

POSITIVE = {'joy', 'love', 'surprise'}
to_polarity = lambda label: 'positive' if label in POSITIVE else 'negative'

test_polarity = test_df['label'].map(to_polarity)
vader_scores = test_df['text'].apply(lambda t: vader.polarity_scores(t)['compound'])
vader_pred = np.where(vader_scores >= 0, 'positive', 'negative')

vader_acc = accuracy_score(test_polarity, vader_pred)
print(f'VADER positive/negative accuracy on test set: {vader_acc:.3f}')

## 5. Classic machine-learning models (TF-IDF features)
**TF-IDF** turns each sentence into numbers. Words that are common in one sentence but rare across all sentences get higher scores. I use single words and word pairs (*bigrams*), so phrases like "not happy" are kept together.

I train three models. For each, I pick the best setting using the **validation** set only:
- **Naive Bayes:** a simple probability model, fast and often a strong baseline for text.
- **Linear SVM:** finds the boundary that best separates the emotions.
- **Logistic Regression:** a linear model that also outputs probabilities.

In [ ]:
def make_features(column):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    X_train = vec.fit_transform(train_df[column])   # learn vocabulary from training data only
    X_val = vec.transform(val_df[column])
    X_test = vec.transform(test_df[column])
    return vec, X_train, X_val, X_test

MODELS = {
    'Naive Bayes':         (lambda p: MultinomialNB(alpha=p),                    [0.01, 0.05, 0.1, 0.5, 1.0]),
    'Linear SVM':          (lambda p: LinearSVC(C=p, class_weight='balanced'),    [0.1, 0.3, 1.0, 3.0]),
    'Logistic Regression': (lambda p: LogisticRegression(C=p, max_iter=2000, class_weight='balanced'), [1.0, 3.0, 10.0, 30.0]),
}

def tune_and_test(column):
    vec, X_train, X_val, X_test = make_features(column)
    rows, fitted = [], {}
    for name, (build, params) in MODELS.items():
        best_p, best_f1 = None, -1
        for p in params:                                  # choose setting on validation set
            m = build(p).fit(X_train, train_df['label'])
            f1 = f1_score(val_df['label'], m.predict(X_val), average='macro')
            if f1 > best_f1: best_p, best_f1 = p, f1
        m = build(best_p).fit(X_train, train_df['label'])  # final model with best setting
        start = time.time(); pred = m.predict(X_test); secs = time.time() - start
        rows.append({'Model': name, 'Best setting': best_p,
                     'Accuracy': accuracy_score(test_df['label'], pred),
                     'Macro F1': f1_score(test_df['label'], pred, average='macro')})
        fitted[name] = (m, pred)
    return vec, pd.DataFrame(rows), fitted

vec_keep, res_keep, fitted_keep = tune_and_test('clean_keep')
vec_nostop, res_nostop, fitted_nostop = tune_and_test('clean_nostop')

comparison = res_keep[['Model']].copy()
comparison['Macro F1 (keep stopwords)'] = res_keep['Macro F1']
comparison['Macro F1 (remove stopwords)'] = res_nostop['Macro F1']
comparison.round(3)

**Stopword experiment:** compare the two columns above. If keeping stopwords scores higher, it shows that the standard cleaning step from my original plan actually throws away useful emotional signal, like negations and intensifiers. I use whichever version scores better from here on.

In [ ]:
use_keep = res_keep['Macro F1'].mean() >= res_nostop['Macro F1'].mean()
CLEAN_COL = 'clean_keep' if use_keep else 'clean_nostop'
results_classic = res_keep if use_keep else res_nostop
fitted = fitted_keep if use_keep else fitted_nostop
vectorizer = vec_keep if use_keep else vec_nostop
print('Using:', 'keep stopwords' if use_keep else 'remove stopwords')
results_classic.round(3)

## 6. Neural network: Bidirectional LSTM
TF-IDF ignores word order. An **LSTM** is a type of recurrent neural network that reads a sentence word by word and remembers context. *Bidirectional* means it reads both forwards and backwards. This builds on the RNN work from my coursework.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
tf.random.set_seed(SEED)

label_to_id = {l: i for i, l in enumerate(LABELS)}
y_train = train_df['label'].map(label_to_id).values
y_val = val_df['label'].map(label_to_id).values
y_test = test_df['label'].map(label_to_id).values

MAX_WORDS, MAX_LEN = 15000, 50
text_vec = layers.TextVectorization(max_tokens=MAX_WORDS, output_sequence_length=MAX_LEN)
text_vec.adapt(train_df['clean_keep'].values)

def to_ids(series):
    return text_vec(tf.constant(series.values)).numpy()

Xtr, Xva, Xte = to_ids(train_df['clean_keep']), to_ids(val_df['clean_keep']), to_ids(test_df['clean_keep'])

# Give rare emotions more weight during training (handles imbalance)
freq = np.bincount(y_train, minlength=len(LABELS))
class_weight = {i: len(y_train) / (len(LABELS) * f) for i, f in enumerate(freq)}

lstm = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(MAX_WORDS, 100, mask_zero=True),     # word -> 100-number vector
    layers.Bidirectional(layers.LSTM(64)),                 # read sentence both ways
    layers.Dropout(0.4),                                   # reduce overfitting
    layers.Dense(len(LABELS), activation='softmax'),
])
lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

stop = callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = lstm.fit(Xtr, y_train, validation_data=(Xva, y_val), epochs=15, batch_size=64,
                   class_weight=class_weight, callbacks=[stop], verbose=2)

lstm_pred = np.array(LABELS)[lstm.predict(Xte, verbose=0).argmax(axis=1)]
lstm_row = {'Model': 'BiLSTM', 'Best setting': f'{len(history.history["loss"])} epochs',
            'Accuracy': accuracy_score(test_df['label'], lstm_pred),
            'Macro F1': f1_score(test_df['label'], lstm_pred, average='macro')}
print(lstm_row)

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(history.history['loss'], label='training loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('BiLSTM learning curve'); plt.legend()
plt.tight_layout(); plt.show()

If validation loss starts rising while training loss keeps falling, the model is **overfitting** (memorising instead of learning). Early stopping keeps the best epoch.

## 7. (Optional, needs GPU) Transformer: DistilBERT
**DistilBERT** is a smaller version of BERT, a language model pre-trained on huge amounts of English text. It already "understands" a lot of language, so it only needs fine-tuning on our emotion data. This is the current standard approach, so it shows how far the simpler models are from the state of the art.

Turn on a GPU first (**Runtime → Change runtime type → T4 GPU**). It takes about 5–10 minutes. Without a GPU, this cell skips itself.

In [ ]:
transformer_row = None
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

if not HAS_GPU:
    print('No GPU found, skipping DistilBERT. (Turn on a T4 GPU in Colab to run it.)')
else:
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)
    CKPT = 'distilbert-base-uncased'
    tok = AutoTokenizer.from_pretrained(CKPT)

    def to_hf(df):
        d = Dataset.from_pandas(pd.DataFrame({'text': df['text'], 'label': df['label'].map(label_to_id)}))
        return d.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=64), batched=True)

    hf_train, hf_val, hf_test = to_hf(train_df), to_hf(val_df), to_hf(test_df)
    bert = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=len(LABELS))

    args = TrainingArguments(output_dir='bert_out', num_train_epochs=2, per_device_train_batch_size=64,
                             per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01,
                             report_to='none', seed=SEED)
    trainer = Trainer(model=bert, args=args, train_dataset=hf_train, eval_dataset=hf_val)
    trainer.train()

    logits = trainer.predict(hf_test).predictions
    bert_pred = np.array(LABELS)[logits.argmax(axis=1)]
    transformer_row = {'Model': 'DistilBERT (fine-tuned)', 'Best setting': '2 epochs',
                       'Accuracy': accuracy_score(test_df['label'], bert_pred),
                       'Macro F1': f1_score(test_df['label'], bert_pred, average='macro')}
    print(transformer_row)

## 8. Results: all models on the test set

In [ ]:
rows = results_classic[['Model', 'Accuracy', 'Macro F1']].to_dict('records') + [lstm_row]
if transformer_row: rows.append(transformer_row)
results = pd.DataFrame(rows)[['Model', 'Accuracy', 'Macro F1']].sort_values('Macro F1', ascending=False)
results = results.reset_index(drop=True)
print(f'VADER (positive vs negative only, not comparable to 6-class scores): accuracy {vader_acc:.3f}')
results.round(3)

In [ ]:
ax = results.set_index('Model')[['Accuracy', 'Macro F1']].plot(kind='barh', figsize=(7, 3.5), xlim=(0.5, 1.0))
ax.set_title('Test-set performance (6 emotions)'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

### 8.1 Fair comparison with VADER
To compare like with like, I collapse my best classic model's six-emotion predictions into positive/negative and measure it the same way as VADER.

In [ ]:
best_classic = results_classic.sort_values('Macro F1', ascending=False).iloc[0]['Model']
best_classic_pred = fitted[best_classic][1]
clf_polarity_acc = accuracy_score(test_polarity, pd.Series(best_classic_pred).map(to_polarity))
print(f'VADER positive/negative accuracy:          {vader_acc:.3f}')
print(f'{best_classic} positive/negative accuracy: {clf_polarity_acc:.3f}')

## 9. Error analysis: which emotions get confused?
A **confusion matrix** shows, for each true emotion (rows), what the model predicted (columns). The diagonal is correct. Anything off the diagonal is a mistake.

In [ ]:
best_pred = best_classic_pred
cm = confusion_matrix(test_df['label'], best_pred, labels=LABELS, normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion matrix: {best_classic} (row-normalised)')
plt.tight_layout(); plt.show()

print(classification_report(test_df['label'], best_pred, labels=LABELS, digits=3))

In [ ]:
# Most common mistakes
errors = pd.DataFrame({'text': test_df['text'].values, 'true': test_df['label'].values, 'pred': best_pred})
errors = errors[errors['true'] != errors['pred']]
print('Most frequent confusions:')
print(errors.groupby(['true', 'pred']).size().sort_values(ascending=False).head(6).to_string())
print()
print('Example mistakes:')
for _, r in errors.sample(min(8, len(errors)), random_state=SEED).iterrows():
    print(f'  TRUE={r.true:<8} PRED={r.pred:<8} | {r.text}')

**What to look for:** the rare emotions (*love*, *surprise*) usually score lowest. *Love* is often confused with *joy* (both positive and warm), and *surprise* with *fear* (both high-arousal). Some errors are arguably **label problems**: the sentence could fairly be read either way. This is a known limitation of emotion datasets, and a real research question: emotions overlap, but the dataset forces one label per sentence.

### 9.1 Which words does the model rely on?
For a linear model, each word has a weight per emotion. The top words show what the model has learned, and whether it makes sense.

In [ ]:
lin_name = 'Logistic Regression'
lin_model = fitted[lin_name][0]
vocab = np.array(vectorizer.get_feature_names_out())
for i, emo in enumerate(lin_model.classes_):
    top = vocab[np.argsort(lin_model.coef_[i])[-8:][::-1]]
    print(f'{emo:<9}: ' + ', '.join(top))

## 10. From emotion to music: the recommender
This connects the emotion model back to Harmony Hub. My literature review described the **arousal–valence model** of emotion:
- **Valence** = how positive the feeling is (Spotify's `valence`, 0–1)
- **Arousal** = how energetic it is (Spotify's `energy`, 0–1)

Each emotion is placed on this map. The system then finds songs whose audio features are closest to that point (**content-based filtering** with nearest neighbours).

There are two modes, because a wellbeing app has a design choice to make:
- **match:** songs that match how you feel (sad → gentle, sad songs)
- **uplift:** songs that gently move you towards a calmer or more positive state (sad → warm, calm songs)

**Note:** these target values are my own assumptions based on the arousal–valence model. They are not validated with real users. That is the main limitation here (see section 11).

In [ ]:
def load_spotify():
    url = 'https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset/resolve/main/dataset.csv'
    try:
        df = pd.read_csv(url)
    except Exception as e:
        print('Download failed:', e)
        print('Download "Spotify Tracks Dataset" from Kaggle and upload dataset.csv to Colab.')
        df = pd.read_csv('dataset.csv')
    df = df.dropna(subset=['track_name', 'artists'])
    df = df.drop_duplicates(subset=['track_name', 'artists'])
    return df.reset_index(drop=True)

songs = load_spotify()
print(len(songs), 'unique tracks')
songs[['track_name', 'artists', 'track_genre', 'valence', 'energy', 'acousticness', 'tempo']].head()

In [ ]:
# Emotion -> target point on the arousal-valence map (valence, energy, acousticness)
MOOD_TARGETS = {
    'match': {
        'sadness':  (0.20, 0.30, 0.70),
        'joy':      (0.85, 0.75, 0.20),
        'love':     (0.70, 0.45, 0.50),
        'anger':    (0.30, 0.90, 0.10),
        'fear':     (0.25, 0.55, 0.40),
        'surprise': (0.70, 0.80, 0.20),
    },
    'uplift': {   # move gently towards calm/positive rather than amplifying negative states
        'sadness':  (0.55, 0.40, 0.60),
        'joy':      (0.85, 0.75, 0.20),
        'love':     (0.70, 0.45, 0.50),
        'anger':    (0.50, 0.30, 0.70),
        'fear':     (0.60, 0.30, 0.70),
        'surprise': (0.70, 0.60, 0.30),
    },
}
FEATURES = ['valence', 'energy', 'acousticness']

def recommend(user_text, mode='uplift', genre=None, n=5, model_name=None):
    model_name = model_name or best_classic
    model = fitted[model_name][0]
    emotion = model.predict(vectorizer.transform([clean(user_text, remove_stopwords=not use_keep)]))[0]

    pool = songs if genre is None else songs[songs['track_genre'].str.contains(genre, case=False, na=False)]
    if pool.empty:
        print(f'No songs found for genre "{genre}", using all genres.'); pool = songs
    # Prefer reasonably popular tracks so results are recognisable
    pool = pool[pool['popularity'] >= pool['popularity'].quantile(0.5)]

    target = np.array([MOOD_TARGETS[mode][emotion]])
    nn = NearestNeighbors(n_neighbors=min(n, len(pool))).fit(pool[FEATURES].values)
    _, idx = nn.kneighbors(target)
    recs = pool.iloc[idx[0]][['track_name', 'artists', 'track_genre'] + FEATURES]

    print(f'You wrote: "{user_text}"')
    print(f'Detected emotion: {emotion.upper()}   |   mode: {mode}' + (f'   |   genre: {genre}' if genre else ''))
    return recs.reset_index(drop=True)

In [ ]:
recommend("i feel so tired and lonely today, nothing is going right", mode='match')

In [ ]:
recommend("i feel so tired and lonely today, nothing is going right", mode='uplift')

In [ ]:
recommend("i just got the job offer, i am so excited!!", genre='pop')

In [ ]:
recommend("i am so angry at how they treated me", mode='uplift')

**Try your own:** change the sentence, the `mode` ('match' or 'uplift') and the `genre` (e.g. 'acoustic', 'hip-hop', 'k-pop', 'indian', 'classical').

In [ ]:
recommend("write how you feel here", mode='uplift')

### 10.1 Sanity check: do recommended songs sit where they should?
For each emotion, I check the average valence and energy of the recommended songs against the target. This only checks that the system does what it was designed to do. It does **not** prove that users feel better, which would need a user study.

In [ ]:
check_rows = []
for mode in ['match', 'uplift']:
    for emo, (v, e, a) in MOOD_TARGETS[mode].items():
        nn = NearestNeighbors(n_neighbors=20).fit(songs[FEATURES].values)
        _, idx = nn.kneighbors([[v, e, a]])
        got = songs.iloc[idx[0]][FEATURES].mean()
        check_rows.append({'mode': mode, 'emotion': emo, 'target valence': v, 'avg valence': got['valence'],
                           'target energy': e, 'avg energy': got['energy']})
pd.DataFrame(check_rows).round(2)

## 11. Findings, limitations and next steps

**Findings** (fill in from your own run):
- Best model: ______ with macro F1 of ______ on the test set.
- Keeping vs removing stopwords: ______.
- VADER can only separate positive from negative, and scored ______, compared with ______ for my model on the same task. Lexicon tools are not enough for a mood-based app.
- Hardest emotions: ______ (most confused with ______).

**Limitations**
1. **Single-label data:** each sentence has exactly one emotion, but real feelings are often mixed.
2. **Dataset style:** sentences mostly start "i feel…", which is easier than real user messages. Performance on real app input would likely be lower.
3. **Emotion → music mapping is assumed:** the valence/energy targets come from theory, not from users. Whether "uplift" songs actually improve mood is untested.
4. **No personalisation:** two people with the same emotion get the same songs. Their own taste and history are ignored.
5. **Music only:** the original Harmony Hub idea also covered films, podcasts and stories.

**Next steps (questions I would like to research further)**
- Detect **mixed emotions** (multi-label) and **intensity**, not just one label.
- Learn the emotion → content mapping **from user feedback**, instead of fixing it by hand.
- Combine emotion with **personal taste** (collaborative filtering).
- Extend to **multiple media types** (music, film, podcasts) in one model.
- Run a small **user study** to measure whether recommendations help mood.